In [1]:
import json
import pennylane as qp
import pennylane.numpy as np

In [ ]:
# A three-qubit state is prepared for you. You do not need to change this.
def prepare(angles):
    """Prepares the state whose Pauli expectation values you have to measure.

    Args:
        angles (list[float]): four rotation angles.
    """
    qp.RY(angles[0], wires=0)
    qp.RX(angles[1], wires=1)
    qp.RY(angles[2], wires=2)
    qp.CNOT(wires=[0, 1])
    qp.RZ(angles[3], wires=1)
    qp.CNOT(wires=[1, 2])
    qp.RX(angles[3], wires=0)


def measure_pauli_word(word, angles):
    """Measures a Pauli word on the prepared state using only computational-basis
    probabilities.

    Args:
        word (str): three characters, each one of "I", "X", "Y", "Z". Character w
            is the Pauli factor acting on wire w.
        angles (list[float]): the four angles handed to prepare().

    Returns:
        tuple[list[str], list[int], float]:
            - the basis change applied to each wire, "none", "H" or "SH";
            - the eight signs, one per computational-basis outcome;
            - the expectation value of the Pauli word.
    """

        # --- Step 1: pick a basis change for each of the three wires -------------
    # Each letter maps to exactly one basis change:
    #     "I" -> "none"     "Z" -> "none"     "X" -> "H"     "Y" -> "SH"
    # Hint: A dictionary is the tidiest way to write a fixed lookup like this one.
    basis = []
    for _, w in enumerate(word):
        if w == "I" or w == "Z":
            basis.append("none")
        elif w == "X":
            basis.append("H")
        else:
            basis.append("SH")

    # --- Step 2: run one circuit and read the eight probabilities ------------
    dev = qp.device("default.qubit", wires=3)
    @qp.qnode(dev)
    def circuit(angles, basis):
        prepare(angles)
        for i, w in enumerate(basis):
            if w == "H":
                qp.Hadamard(wires=i)
            elif w == "SH":
                qp.adjoint(qp.S(wires=i))
                qp.Hadamard(wires=i)
        return qp.probs(wires=range(3))
        
    probabilities = circuit(angles, basis)
        
    # --- Step 3: work out the sign of each of the eight outcomes -------------
    signs = []
    
    active_wires = [w for w, char in enumerate(word) if char != "I"]

    for b in range(8):
        count = 0
        for w in active_wires:
            bit = (b >> (2 - w)) & 1

            if bit == 1:
                count += 1

        sign = (-1) ** count

        signs.append(sign)

    # --- Step 4: combine them into the expectation value ---------------------
    value = sum([probabilities[i]*signs[i] for i in range(8)])

    return basis, signs, value